# MQTTGuard: reproduce the verified ML and ESP32 simulation version

CPU-only Colab entry point for the existing MQTTGuard experiment. This notebook adopts the teammate's Colab workflow while preserving the current firmware, distinct gap clip, diagnostics and report definitions.

**90.35% accuracy and 83.87% SlowITe F1 describe the same classifier.** The primary dataset split is train captures 1+2, validate 3, test 4. This is a reproduction, not a new independent validation study.

Run top to bottom in a fresh Colab runtime. No project ZIP upload, GPU or API key is required. The source commit below is fixed so future changes to GitHub main cannot silently change this experiment. Internet access is required for source, dependencies and the official dataset. The notebook is checked locally; a hosted execution is not claimed merely because it is present in GitHub.


## 1. Fetch the fixed source revision

Change `SOURCE_COMMIT` only as an explicit, documented version update. Setup reuses a matching checkout without deleting local work.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/ayatinkering/MQTTguard.git'
SOURCE_COMMIT = 'bd0022fd536751ce4e304db0df62f3507b11ffd6'
ROOT = Path('/content') / ('MQTTGuard-repro-' + SOURCE_COMMIT[:8])

if not (ROOT / '.git').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'Use a fresh runtime or choose an empty checkout directory: {ROOT}')
    ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'init', str(ROOT)], check=True)
    subprocess.run(['git', 'remote', 'add', 'origin', REPOSITORY], cwd=ROOT, check=True)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=ROOT, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
assert actual == SOURCE_COMMIT, (actual, SOURCE_COMMIT)
print('Source commit:', actual)
print('Checkout:', ROOT)


## 2. Install locked dependencies

The pipeline runs in fresh subprocesses so preloaded Colab NumPy/scikit-learn modules do not affect training. `g++` is required for the real C++ export parity check.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.lock')], check=True)
for command in ['g++', 'curl', 'git']:
    subprocess.run([command, '--version'], check=True, stdout=subprocess.DEVNULL)
print('Locked dependencies installed; required tools available.')


## 3. Download and audit original captures

The repository downloader verifies publisher hashes. Training uses the four separate raw captures, not a shuffled or balanced processed CSV. The cleaned audit file is not additional training data.


In [ ]:
subprocess.run([sys.executable, 'scripts/download_dataset.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/inspect_dataset.py'], cwd=ROOT, check=True)
for capture in range(1, 5):
    path = ROOT / 'data/raw' / f'MQTTEEB-D_dataset_loop_{capture}.csv'
    assert path.is_file(), path
    print(path.name, f'{path.stat().st_size:,} bytes')


## 4. Train, export and regenerate all report artifacts

The four features are `count`, `mean_tcp_len`, `mean_interarrival_s`, `mean_mqtt_kalive` in that exact order. Singleton interarrival is missing, imputed using training-only medians. Preserve empty windows and chronological IDs.

Training evaluates the compact forest candidates, the count baseline, capture-held-out robustness folds and ideal-label opportunity diagnostic. It exports the selected primary classifier and checks Python/C++ parity, including threshold-boundary inputs.

Publishing regenerates CSVs, both figures and RESULTS.md. Bundling retains current diagnostics and creates the source-build fingerprint. This does not update the online Wokwi project.


In [ ]:
for script in ['train_pipeline.py', 'publish_results.py', 'bundle_firmware.py']:
    print('Running', script)
    subprocess.run([sys.executable, 'scripts/' + script], cwd=ROOT, check=True)


## 5. Run automated and recorded-evidence checks

These checks compare saved evidence against the regenerated build/model. They do not constitute a new online Wokwi run. A mismatch must be investigated, not removed to obtain a passing notebook.


In [ ]:
for arguments in [
    ['-m', 'pytest', 'scripts', '-q'],
    ['scripts/verify_demo_evidence.py'],
    ['scripts/verify_wokwi_serial.py', 'docs/evidence/wokwi-serial-full.txt'],
    ['scripts/verify_demo_evidence.py', '--file', 'docs/evidence/mqtt_recovery.jsonl',
     '--output', 'reports/colab_mqtt_recovery_verification.json'],
]:
    subprocess.run([sys.executable] + arguments, cwd=ROOT, check=True)


## 6. Read the results with their denominators

Classification metrics cover only pure legitimate/SlowITe windows in the primary held-out capture. Detecting a label-run proxy is different from correctly classifying every attack window.

The persistence CSV's `base_classifier_window_FPR` is the pre-confirmation classifier FPR, repeated across k. Examine clean false starts, confirmed benign windows, excluded event starts, missed runs and delay together. Capture 4 has no adjacent pure-benign windows and cannot establish false-alarm suppression. Read robustness results too; each fold has a separately trained model from the same testbed/day.


In [ ]:
import json
from IPython.display import Image, display

for name in ['classification_metrics.csv', 'persistence_metrics.csv']:
    print(name)
    print((ROOT / 'docs/results' / name).read_text())
manifest = json.loads((ROOT / 'docs/results/manifest.json').read_text())
primary = json.loads((ROOT / 'docs/results/primary.json').read_text())
print('Split:', manifest['split'])
print('Export parity:', manifest['export_parity'])
print('Primary window coverage:', primary[0]['coverage'])
print('Firmware build:', (ROOT / 'firmware/wokwi/build.json').read_text())
display(Image(filename=str(ROOT / 'docs/results/classification_metrics.png')))
display(Image(filename=str(ROOT / 'docs/results/primary_delay.png')))


## 7. Package, verify and download one consistent handoff

The archive includes training source scripts, locked dependencies, model, predictions, result exports, maintainable firmware and the bundled Wokwi sketch. `handoff_manifest.json` records the source commit, dataset hashes, build identity, working-tree differences and every other artifact's SHA-256 hash.

Keep versions together. The online verified build is recorded in the repository evidence. If a new build differs, use a copy of the Wokwi project and collect new target evidence before replacing the verified project. TEST mode bypasses AI; MODEL lines run inference. Never use scripted tests as accuracy measurements.

The notebook itself is available separately in GitHub. It pins the implementation source commit immediately preceding its addition, so a self-referential commit hash is not needed.


In [ ]:
from zipfile import ZipFile
import hashlib

BUNDLE = ROOT.parent / 'MQTTGuard_ML_outputs.zip'
subprocess.run([sys.executable, 'scripts/package_ml_outputs.py', '--output', str(BUNDLE)], cwd=ROOT, check=True)
with ZipFile(BUNDLE) as archive:
    provenance = json.loads(archive.read('handoff_manifest.json'))
    assert provenance['source_commit'] == SOURCE_COMMIT
    assert set(archive.namelist()) == set(provenance['artifact_sha256']) | {'handoff_manifest.json'}
    for name, expected in provenance['artifact_sha256'].items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected, name
print('All archive hashes verified:', len(provenance['artifact_sha256']))
print('ZIP SHA-256:', hashlib.sha256(BUNDLE.read_bytes()).hexdigest())
from google.colab import files
files.download(str(BUNDLE))


## What is still research work?

See `docs/RESEARCH_FINDINGS.md` for completed measurements and pending experiments. Highest priorities are matched-threshold/filter comparisons, the complete alarm-metric audit, sustained benign sessions and independent timestamped MQTT traces. Running this notebook reproduces existing results; it does not complete these extensions. Any new tuning on the already inspected captures is exploratory.
